## **Apa itu Boosting?**

Kalau Bagging itu:

Banyak model belajar secara bersamaan lalu hasilnya digabung.

Boosting kebalikannya. Model dilatih satu per satu. Misalnya ada data seperti ini

Bagging                 |       Boosting

Training paralel	    |       Training berurutan

Semua model independen	|       Model saling memperbaiki

Mengurangi variance	    |       Mengurangi bias

Contoh: Random Forest	|       Gradient Boosting, AdaBoost, XGBoost

### **Studi Kasus: Airbnb**
- Latih model Gradient Boosting.
- Evaluasi MSE atau RMSE.
- Sesuaikan hyperparameter seperti n_estimators dan learning_rate.

#### **Cara Menyelesaikan Studi Kasus**
1. Melatih Gradient Boosting: Melatih model gradient boosting.
2. Mengevaluasi Metrik: Menghitung MSE atau RMSE.
3. Menyetel Hyperparameter: Menyesuaikan n_estimators, learning_rate, dll.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.impute import SimpleImputer

In [5]:
df = pd.read_csv(r'E:\DigitalentDA\Datasets\Airbnb_Data.csv')

df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 74111 entries, 0 to 74110
Data columns (total 29 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      74111 non-null  int64  
 1   log_price               74111 non-null  float64
 2   property_type           74111 non-null  str    
 3   room_type               74111 non-null  str    
 4   amenities               74111 non-null  str    
 5   accommodates            74111 non-null  int64  
 6   bathrooms               73911 non-null  float64
 7   bed_type                74111 non-null  str    
 8   cancellation_policy     74111 non-null  str    
 9   cleaning_fee            74111 non-null  bool   
 10  city                    74111 non-null  str    
 11  description             74111 non-null  str    
 12  first_review            58247 non-null  str    
 13  host_has_profile_pic    73923 non-null  str    
 14  host_identity_verified  73923 non-null  str    
 

,id,log_price,accommodates,bathrooms,latitude,longitude,number_of_reviews,review_scores_rating,bedrooms,beds
count,7.411100e+04,74111.000000,74111.000000,73911.000000,74111.000000,74111.000000,74111.000000,57389.000000,74020.000000,73980.000000
mean,1.126662e+07,4.782069,3.155146,1.235263,38.445958,-92.397525,20.900568,94.067365,1.265793,1.710868
std,6.081735e+06,0.717394,2.153589,0.582044,3.080167,21.705322,37.828641,7.836556,0.852143,1.254142
min,3.440000e+02,0.000000,1.000000,0.000000,33.338905,-122.511500,0.000000,20.000000,0.000000,0.000000
25%,6.261964e+06,4.317488,2.000000,1.000000,34.127908,-118.342374,1.000000,92.000000,1.000000,1.000000
50%,1.225415e+07,4.709530,2.000000,1.000000,40.662138,-76.996965,6.000000,96.000000,1.000000,1.000000
75%,1.640226e+07,5.220356,4.000000,1.000000,40.746096,-73.954660,23.000000,100.000000,1.000000,2.000000
max,2.123090e+07,7.600402,16.000000,8.000000,42.390437,-70.985047,605.000000,100.000000,10.000000,18.000000


In [7]:
df.head()

,id,log_price,property_type,room_type,amenities,accommodates,bathrooms,bed_type,cancellation_policy,cleaning_fee,...,latitude,longitude,name,neighbourhood,number_of_reviews,review_scores_rating,thumbnail_url,zipcode,bedrooms,beds
0,6901257,5.010635,Apartment,Entire home/apt,"{""Wireless Internet"",""Air conditioning"",Kitche...",3,1.0,Real Bed,strict,True,...,40.696524,-73.991617,Beautiful brownstone 1-bedroom,Brooklyn Heights,2,100.0,https://a0.muscache.com/im/pictures/6d7cbbf7-c...,11201,1.0,1.0
1,6304928,5.129899,Apartment,Entire home/apt,"{""Wireless Internet"",""Air conditioning"",Kitche...",7,1.0,Real Bed,strict,True,...,40.766115,-73.989040,Superb 3BR Apt Located Near Times Square,Hell's Kitchen,6,93.0,https://a0.muscache.com/im/pictures/348a55fe-4...,10019,3.0,3.0
2,7919400,4.976734,Apartment,Entire home/apt,"{TV,""Cable TV"",""Wireless Internet"",""Air condit...",5,1.0,Real Bed,moderate,True,...,40.808110,-73.943756,The Garden Oasis,Harlem,10,92.0,https://a0.muscache.com/im/pictures/6fae5362-9...,10027,1.0,3.0
3,13418779,6.620073,House,Entire home/apt,"{TV,""Cable TV"",Internet,""Wireless Internet"",Ki...",4,1.0,Real Bed,flexible,True,...,37.772004,-122.431619,Beautiful Flat in the Heart of SF!,Lower Haight,0,NaN,https://a0.muscache.com/im/pictures/72208dad-9...,94117.0,2.0,2.0
4,3808709,4.744932,Apartment,Entire home/apt,"{TV,Internet,""Wireless Internet"",""Air conditio...",2,1.0,Real Bed,moderate,True,...,38.925627,-77.034596,Great studio in midtown DC,Columbia Heights,4,40.0,NaN,20009,0.0,1.0


In [8]:
df.isnull().sum()

id                            0
log_price                     0
property_type                 0
room_type                     0
amenities                     0
accommodates                  0
bathrooms                   200
bed_type                      0
cancellation_policy           0
cleaning_fee                  0
city                          0
description                   0
first_review              15864
host_has_profile_pic        188
host_identity_verified      188
host_response_rate        18299
host_since                  188
instant_bookable              0
last_review               15827
latitude                      0
longitude                     0
name                          0
neighbourhood              6872
number_of_reviews             0
review_scores_rating      16722
thumbnail_url              8216
zipcode                     966
bedrooms                     91
beds                        131
dtype: int64

In [9]:
df['bathrooms'] = df['bathrooms'].fillna(df['bathrooms'].median())
df['review_scores_rating'] = df['review_scores_rating'].fillna(df['review_scores_rating'].median())
df['bedrooms'] = df['bedrooms'].fillna(df['bedrooms'].median())
df['beds'] = df['beds'].fillna(df['beds'].median())


imputer = SimpleImputer(strategy="most_frequent")

df['first_review'] = imputer.fit_transform(df[["first_review"]]).ravel()
df['host_has_profile_pic'] = imputer.fit_transform(df[["host_has_profile_pic"]]).ravel()
df['host_identity_verified'] = imputer.fit_transform(df[["host_identity_verified"]]).ravel()
df['host_response_rate'] = imputer.fit_transform(df[["host_response_rate"]]).ravel()
df['host_since'] = imputer.fit_transform(df[["host_since"]]).ravel()
df['last_review'] = imputer.fit_transform(df[["last_review"]]).ravel()
df['neighbourhood'] = imputer.fit_transform(df[['neighbourhood']]).ravel()
df['thumbnail_url'] = imputer.fit_transform(df[['thumbnail_url']]).ravel()
df['zipcode'] = imputer.fit_transform(df[['zipcode']]).ravel()

In [10]:
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 74111 entries, 0 to 74110
Data columns (total 29 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      74111 non-null  int64  
 1   log_price               74111 non-null  float64
 2   property_type           74111 non-null  str    
 3   room_type               74111 non-null  str    
 4   amenities               74111 non-null  str    
 5   accommodates            74111 non-null  int64  
 6   bathrooms               74111 non-null  float64
 7   bed_type                74111 non-null  str    
 8   cancellation_policy     74111 non-null  str    
 9   cleaning_fee            74111 non-null  bool   
 10  city                    74111 non-null  str    
 11  description             74111 non-null  str    
 12  first_review            74111 non-null  str    
 13  host_has_profile_pic    74111 non-null  str    
 14  host_identity_verified  74111 non-null  str    
 

,id,log_price,accommodates,bathrooms,latitude,longitude,number_of_reviews,review_scores_rating,bedrooms,beds
count,7.411100e+04,74111.000000,74111.000000,74111.000000,74111.000000,74111.000000,74111.000000,74111.000000,74111.000000,74111.000000
mean,1.126662e+07,4.782069,3.155146,1.234628,38.445958,-92.397525,20.900568,94.503434,1.265467,1.709611
std,6.081735e+06,0.717394,2.153589,0.581386,3.080167,21.705322,37.828641,6.943159,0.851671,1.253389
min,3.440000e+02,0.000000,1.000000,0.000000,33.338905,-122.511500,0.000000,20.000000,0.000000,0.000000
25%,6.261964e+06,4.317488,2.000000,1.000000,34.127908,-118.342374,1.000000,93.000000,1.000000,1.000000
50%,1.225415e+07,4.709530,2.000000,1.000000,40.662138,-76.996965,6.000000,96.000000,1.000000,1.000000
75%,1.640226e+07,5.220356,4.000000,1.000000,40.746096,-73.954660,23.000000,99.000000,1.000000,2.000000
max,2.123090e+07,7.600402,16.000000,8.000000,42.390437,-70.985047,605.000000,100.000000,10.000000,18.000000


In [11]:
categorical_cols = df.select_dtypes(include="object").columns

print(categorical_cols)

Index(['property_type', 'room_type', 'amenities', 'bed_type',
       'cancellation_policy', 'city', 'description', 'first_review',
       'host_has_profile_pic', 'host_identity_verified', 'host_response_rate',
       'host_since', 'instant_bookable', 'last_review', 'name',
       'neighbourhood', 'thumbnail_url', 'zipcode'],
      dtype='str')


C:\Users\yumah\AppData\Local\Temp\ipykernel_29728\343913642.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include="object").columns


In [12]:
for col in categorical_cols:
    print(col, ":", df[col].nunique())

property_type : 35
room_type : 3
amenities : 67122
bed_type : 5
cancellation_policy : 5
city : 6
description : 73479
first_review : 2554
host_has_profile_pic : 2
host_identity_verified : 2
host_response_rate : 80
host_since : 3087
instant_bookable : 2
last_review : 1371
name : 73359
neighbourhood : 619
thumbnail_url : 65883
zipcode : 769


In [13]:
df["host_response_rate"] = (
    df["host_response_rate"]
    .str.replace("%", "", regex=False)
    .astype(int)
)

In [17]:
for col in ["host_since", "first_review", "last_review"]:
    df[col] = pd.to_datetime(df[col], format="%Y-%m-%d")
    df[col] = df[col].dt.year

In [22]:
class_cols = [
    "property_type",
    "room_type",
    "bed_type",
    "cancellation_policy",
    "city",
    "host_has_profile_pic",
    "host_identity_verified",
    "instant_bookable"
]

df = pd.get_dummies(
    df,
    columns=class_cols,
    drop_first=True,
    dtype=int
)

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 74111 entries, 0 to 74110
Data columns (total 67 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   id                                   74111 non-null  int64  
 1   log_price                            74111 non-null  float64
 2   accommodates                         74111 non-null  int64  
 3   bathrooms                            74111 non-null  float64
 4   cleaning_fee                         74111 non-null  bool   
 5   first_review_year                    74111 non-null  int32  
 6   host_response_rate                   74111 non-null  int64  
 7   host_since_year                      74111 non-null  int32  
 8   last_review_year                     74111 non-null  int32  
 9   latitude                             74111 non-null  float64
 10  longitude                            74111 non-null  float64
 11  number_of_reviews                    74

In [32]:
df.rename(columns={
    "first_review": "first_review_year",
    "host_since": "host_since_year",
    "last_review": "last_review_year"
}, inplace=True)

In [33]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 74111 entries, 0 to 74110
Data columns (total 67 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   id                                   74111 non-null  int64  
 1   log_price                            74111 non-null  float64
 2   accommodates                         74111 non-null  int64  
 3   bathrooms                            74111 non-null  float64
 4   cleaning_fee                         74111 non-null  bool   
 5   first_review_year                    74111 non-null  int32  
 6   host_response_rate                   74111 non-null  int64  
 7   host_since_year                      74111 non-null  int32  
 8   last_review_year                     74111 non-null  int32  
 9   latitude                             74111 non-null  float64
 10  longitude                            74111 non-null  float64
 11  number_of_reviews                    74

In [36]:
X = df.drop(columns=['log_price'])
y = df['log_price']

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [56]:
model = GradientBoostingRegressor(
    random_state=42,
    n_estimators=150,
    learning_rate=0.1,
    max_depth=5
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [57]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)

print("===EVALUASI MODEL===")
print("MSE: ", mse)
print("RMSE: ", rmse)
print("MAE: ", mae)
print("R2: ", r2)
print("Training Score: ", train_score)
print("Testing Score: ", test_score)

===EVALUASI MODEL===
MSE:  0.15420810520119002
RMSE:  0.39269339846907286
MAE:  0.286132063183714
R2:  0.6998241559564725
Training Score:  0.7263230636050495
Testing Score:  0.6998241559564725
